# Bài giảng notebook: Source code RAG cho YouTube Video Q&A Assistant

Notebook này giúp bạn tự chạy từng phần của source code RAG trong project. Mỗi phần có giải thích trước và sau cell code để bạn hiểu luồng từ URL YouTube đến câu trả lời có timestamp.

Bạn nên chạy notebook từ thư mục gốc project hoặc chỉnh `PROJECT_ROOT` ở cell đầu tiên.

## 1. Chuẩn bị môi trường

Cell dưới thêm thư mục `backend` vào `sys.path` để notebook import được các module production. Nếu bạn mở notebook từ nơi khác, hãy sửa `PROJECT_ROOT` thành đường dẫn project.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
BACKEND_ROOT = PROJECT_ROOT / "backend"

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

print("Project root:", PROJECT_ROOT)
print("Backend import path ready:", BACKEND_ROOT.exists())

Sau cell này, Python có thể import các service thật trong `backend/app`. Đây là cách tốt để học code production mà không copy logic sang notebook.

## 2. Parse YouTube URL

Service `extract_youtube_video_id` nhận nhiều dạng URL YouTube và trả về video ID 11 ký tự. Bạn hãy thay `YOUTUBE_URL` bằng video có transcript.

In [ ]:
from app.services.extraction.video_url_service import extract_youtube_video_id

YOUTUBE_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
video_id = extract_youtube_video_id(YOUTUBE_URL)
video_id

Nếu URL không thuộc YouTube hoặc video ID sai định dạng, service sẽ raise `ValueError`. Route FastAPI chuyển lỗi này thành HTTP 400.

## 3. Tạo transcript mẫu để học chunking

Để notebook chạy ổn định khi chưa có mạng, phần này dùng transcript mẫu nhỏ. Khi muốn thử video thật, bạn có thể dùng `fetch_transcript(video_id)` ở phần sau.

In [ ]:
from app.schemas.transcript import TranscriptSegment

sample_segments = [
    TranscriptSegment(text="Retrieval augmented generation starts by collecting relevant context.", start_seconds=0, end_seconds=5),
    TranscriptSegment(text="The transcript is cleaned and split into chunks with timestamps.", start_seconds=5, end_seconds=10),
    TranscriptSegment(text="A retriever scores chunks against the user question.", start_seconds=10, end_seconds=15),
    TranscriptSegment(text="The answer is generated from the retrieved context and sources.", start_seconds=15, end_seconds=20),
]

sample_segments

Transcript production dùng cùng schema `TranscriptSegment`, nên logic chunking có thể chạy giống nhau cho dữ liệu mẫu và dữ liệu thật.

## 4. Chia chunk và giữ timestamp

`chunk_transcript` gom nhiều segment thành chunk vừa đủ dài. Mỗi chunk có `chunk_id`, `video_id`, text và khoảng thời gian bắt đầu/kết thúc.

In [ ]:
from app.services.rag.text_processing import chunk_transcript

chunks = chunk_transcript(
    video_id="demo_video_1",
    segments=sample_segments,
    target_words=12,
    overlap_words=3,
)

for chunk in chunks:
    print(chunk.chunk_id, chunk.start_seconds, chunk.end_seconds)
    print(chunk.text)
    print()

Chunk overlap giúp câu hỏi khớp tốt hơn khi ý nghĩa nằm ở ranh giới giữa hai đoạn. Với video dài, bạn có thể tăng `target_words` để giảm số chunk.

## 5. Lưu index local và retrieval

MVP dùng `LocalRagStore`, một retriever BM25-style không cần API key. Nó không phải embedding semantic thật, nhưng đủ để minh họa luồng RAG end-to-end.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path
from app.services.rag.local_store import LocalRagStore

with TemporaryDirectory() as temp_dir:
    store = LocalRagStore(Path(temp_dir) / "index.json")
    store.upsert_video("demo_video_1", chunks)
    results = store.retrieve("demo_video_1", "How does retrieval use transcript context?", top_k=3)

for item in results:
    print(item.score, item.chunk.chunk_id, item.chunk.text)

Kết quả retrieval là danh sách chunk đã xếp theo điểm liên quan. API `/chat/ask` dùng phần này để tạo `sources` trả về frontend.

## 6. Sinh câu trả lời từ context

`generate_answer` hiện tạo câu trả lời extractive bằng cách trích các chunk tốt nhất. Sau MVP, bạn có thể thay hàm này bằng LLM provider.

In [ ]:
from app.services.rag.generation_service import generate_answer

question = "How does retrieval use transcript context?"
answer = generate_answer(question, results)
print(answer)

Đây là generation layer. Route không cần biết câu trả lời đến từ extractive generator, OpenAI hay model local; chỉ cần nhận `answer` và `sources`.

## 7. Chạy với transcript thật

Cell này cần mạng và video phải có transcript. Nếu lỗi, hãy thử video khác hoặc chạy lại khi có kết nối ổn định.

In [ ]:
from app.services.extraction.transcript_service import fetch_transcript

# Bỏ comment để thử transcript thật.
# real_segments, language_code = fetch_transcript(video_id)
# real_chunks = chunk_transcript(video_id, real_segments)
# print(language_code, len(real_segments), len(real_chunks))

Khi cell thật chạy được, bạn có thể truyền `real_chunks` vào `LocalRagStore` giống phần demo để tự kiểm tra toàn bộ pipeline.